# YOLO11m - Colab

Train, evaluate, and run inference with YOLO11m on Google Drive data.
Setup cells mirror `colab_template.ipynb`.

In [ ]:
REPO = "road-damage-detection"

# Clone the repository (skip if already cloned)
!test -d $REPO || git clone https://github.com/orzmik/road-damage-detection.git
%cd $REPO

# Install dependencies needed for Colab runs
!pip -q install ultralytics wandb

In [ ]:
BRANCH = "feature/yolo_11m-training"

!git fetch origin
!git checkout $BRANCH
!git pull origin $BRANCH

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

In [ ]:
# My Drive:
DRIVE_ROOT = "MyDrive/road_damage_detection"

# Shared drive (teammates):
# DRIVE_ROOT = "Shareddrives/<TeamDrive>/road_damage_detection"

In [ ]:
from src.config.colab.drive import DriveConfig, ensure_drive_paths, mount_drive

mount_drive()

drive_cfg = DriveConfig(drive_root=DRIVE_ROOT)
paths = ensure_drive_paths(drive_cfg)

print(f"Drive root:       {paths.root}")
print(f"Processed data:   {paths.processed_yolo}")
print(f"Models:           {paths.models}")
print(f"Training runs:    {paths.runs}")
print(f"Wroclaw images:   {paths.wroclaw_images}")

## Training

In [ ]:
import wandb
from google.colab import userdata


wandb.login(key=userdata.get('WANDB_API_KEY'))

In [ ]:
from src.config.yolo import create_yolo_data_yaml

import shutil
from pathlib import Path

LOCAL_DATA = Path("/content/data/processed-yolo")
DRIVE_DATA = paths.processed_yolo
if not LOCAL_DATA.exists():
    print("Copying dataset from Drive to local disk (one-time per session)...")
    shutil.copytree(DRIVE_DATA, LOCAL_DATA)
    print("Done.")
else:
    print("Local copy already exists, skipping copy.")

data_yaml = create_yolo_data_yaml(
    Path("/content/data/road_damage_local.yaml"),
    LOCAL_DATA,
)

In [ ]:
from src.config.wandb import WandbConfig
from src.config.yolo import train_yolo

wandb_cfg = WandbConfig(
    project="road-damage-classification",
    entity="project-nn",
    name="yolo11m_colab",
    job_type="train",
    config={
        "model": "yolo11m",
        "epochs": 50,
        "imgsz": 640,
        "batch": 16,
    },
)

# If CUDA OOM on T4, reduce batch to 8 (keep other hyperparameters unchanged).
train_out = train_yolo(
    weights="yolo11m.pt",
    data_yaml=data_yaml,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=8,
    device=0,
    project_dir=paths.runs,
    run_name="yolo11m_colab",
    wandb_cfg=wandb_cfg,
)

train_out.save_dir

## Evaluation (test set)

In [ ]:
from src.config.yolo import evaluate_yolo

best_weights = train_out.save_dir / "weights" / "best.pt"
metrics = evaluate_yolo(
    weights=best_weights,
    data_yaml=data_yaml,
    split="test",
    device=0,
    project_dir=paths.runs,
    run_name="yolo11m_colab_test",
)
metrics

## Inference (Wroclaw images)

In [ ]:
from src.config.yolo import predict_yolo

predict_results = predict_yolo(
    weights=best_weights,
    source=paths.wroclaw_images,
    device=0,
    project_dir=paths.runs,
    run_name="yolo11m_colab_infer",
    save=True,
)
predict_results